In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 21:27:15.013378: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 21:27:15.921934: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'],
        "ports": [50151, 50152, 50153]
      }
      
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 21:27:17,156 [DEBUG] [Rain] Rain is initialized
2023-07-04 21:27:17,158 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 21:27:17,160 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/coord/
2023-07-04 21:27:17,162 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 21:27:17,163 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 21:27:17,165 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/divider/
2023-07-04 21:27:17,167 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/divider/
2023-07-04 21:27:17,169 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 21:27:17,177 [DEBUG] [Rain] Creating workers
2023-07-04 21:27:17,184 [INFO] [Provisioner] provisioner is serving
2023-07-04 21:27:17,186 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 21:27:17,188 [INFO] [Coordinator] coordinator is serving
2023-07-04 21:27:17,189 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 21:27:17,195 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 21:27:17,197 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 21:27:17,198 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 21:27:17,199 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/worker/
2023-07-04 21:27:17,201 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 21:27:17,203 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/worker/
2023-07-04 21:27:17,205 [INFO] [Worker_

157/157 [==============================] - 3s 9ms/step - loss: 0.7208 - accuracy: 0.7752
sending data to divider


2023-07-04 21:27:40,727 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}


157/157 [==============================] - 3s 9ms/step - loss: 0.7115 - accuracy: 0.7766


2023-07-04 21:27:40,730 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


...

2023-07-04 21:27:48,698 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:27:48,702 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider
157/157 [==============================] - 3s 8ms/step - loss: 0.2417 - accuracy: 0.9291


2023-07-04 21:27:48,741 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}


sending data to divider


2023-07-04 21:27:48,746 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-04 21:27:49,027 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 21:27:49,038 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
DEBUG:DeepLearning:Asynchronous update is done by worker 3
2023-07-04 21:27:49,054 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 21:27:49,075 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
DEBUG:DeepLearning:Asynchronous update is done by wor

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1345 - accuracy: 0.9577

Test accuracy: 95.8%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-04 21:27:49,491 [DEBUG] [Rain] Creating workers
DEBUG:Rain:Creating workers
2023-07-04 21:27:49,495 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-04 21:27:49,496 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-04 21:27:49,498 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-04 21:27:49,499 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-04 21:27:49,501 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 21:27:49,503 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-04 21:27:49,

157/157 [==============================] - 3s 8ms/step - loss: 0.2092 - accuracy: 0.9380
sending data to divider


2023-07-04 21:28:11,294 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:28:11,296 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
2023-07-04 21:28:11,305 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:28:11,309 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3


sending data to divider
157/157 [==============================] - 3s 8ms/step - loss: 0.2065 - accuracy: 0.9388


2023-07-04 21:28:11,352 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:28:11,354 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


...

2023-07-04 21:28:17,660 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:28:17,663 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider
157/157 [==============================] - 2s 5ms/step - loss: 0.1606 - accuracy: 0.9535


2023-07-04 21:28:17,680 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:28:17,683 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1


sending data to divider


2023-07-04 21:28:17,712 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-04 21:28:17,865 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 21:28:17,869 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-04 21:28:17,893 [DEBUG] [DeepLearning] Iteration 3/3 complete.
DEBUG:DeepLearning:Iteration 3/3 complete.
2023-07-04 21:28:17,894 [ERROR] [DividerAmbassador] Error stopping serving: '_ServerState' object has no attribute 'name'
ERROR:DividerAmbassador:Error stopping serving: '_Server

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0985 - accuracy: 0.9705

Test accuracy: 97.0%
